In [ ]:
import subprocess
import win32gui
import win32con
import time
import mss
import cv2
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
import pyautogui
import win32api
from ctypes import windll
import win32ui
import random
import win32process
import socket
import ctypes
import os
import platform
import psutil
import keyboard
from sdlarch_rl.utils.utils import get_latest_model, FrameSkip, AugmentObservation
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from sdlarch_rl.utils.stf6 import STF6Env
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from pathlib import Path
from stable_baselines3 import PPO

# env = STF6Env()
SAVE_DIR="./model-sf4-all"
DETERMINISTIC=False

SAVE_DIR = Path(SAVE_DIR)

latest_model_path = str(SAVE_DIR) + "/best_model"

print("loading from: " + str(latest_model_path))

global myEnv

def make_env():
    def _init():
        global myEnv
        myEnv = STF6Env()

        # env = AugmentObservation(myEnv)
        env = WarpFrame(myEnv, width=48, height=48)
        env = WarpFrame(myEnv, width=96, height=96)
        # env = FrameSkip(env, skip=4)
        
        return env
    return _init

env = make_vec_env(make_env(), n_envs=1, vec_env_cls=DummyVecEnv)
env = VecFrameStack(env, 4, channels_order='last')

model = PPO.load(
    str(latest_model_path), 
    env=env, 
    verbose=0, 
)

SCREEN_WIDTH = 640*3
SCREEN_HEIGHT = 480*3

prev_keys = set()

env.reset()

could_use_ai = False

color=(0, 0, 255)

while True:
    # frame_start = time.perf_counter()

    action = [np.zeros(16, dtype=np.uint8)]


    if keyboard.is_pressed('k'):
        could_use_ai = not could_use_ai

        if could_use_ai:
            color=(0, 255, 0)
        else:
            color=(0, 0, 255)
        
        time.sleep(0.5)

        print("k is pressed ", could_use_ai)

    # libretro
    # ["B", "Y", "SELECT", "START", "UP", "DOWN", "LEFT", "RIGHT", "A", "X", "L1", "R1", "L2", "R2", "L3", "R3"]

    if not could_use_ai:
        # dpad
        if keyboard.is_pressed('up'):
            action[0][4] = 1
        if keyboard.is_pressed('down'):
            action[0][5] = 1
        if keyboard.is_pressed('left'):
            action[0][6] = 1
        if keyboard.is_pressed('right'):
            action[0][7] = 1
    
        # buttons
        if keyboard.is_pressed('i'):
            action[0][8] = 1
        if keyboard.is_pressed('o'):
            action[0][10] = 1
        if keyboard.is_pressed('p'): # strong kick # 11
            action[0][11] = 1
    else:
        # action[0] = np.random.choice([0, 1], size=16)
        action, _ = model.predict(obs, deterministic=DETERMINISTIC)
        # action[0] = a

    obs, _, _, _ = env.step(action)

    img = myEnv.render()

    if img is not None:
        img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        
        cv2.circle(img, center=(100, 100), radius=50, color=color, thickness=2)
        
        cv2.imshow("game", img)
    
        cv2.waitKey(1)

# s.close()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


loading from: model-sf4-all/best_model
Window founded HWND: 8913482.


D:\Python311\Lib\site-packages\stable_baselines3\common\save_util.py:166: UserWarning: Could not deserialize object clip_range. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute 'FloatSchedule' on <module 'stable_baselines3.common.utils' from 'D:\\Python311\\Lib\\site-packages\\stable_baselines3\\common\\utils.py'>
  warnings.warn(
D:\Python311\Lib\site-packages\stable_baselines3\common\save_util.py:166: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute 'FloatSchedule' on <module 'stable_baselines3.common.utils' from 'D:\\Python311\\Lib\\site-packages\\stable_baselines3\\common\\utils.py'>
  warnings.warn(
D:\Python311\Lib\site-packages\stable_baselines3\common\save_util.py:449: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to c

reset obs shape (128, 228, 3)
k is pressed  True
k is pressed  False
k is pressed  True
